In [19]:
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

# =========================
# 1. Load input files
# =========================

# All lncRNAs
lnc = pd.read_csv("../../data/LPI/human/lncRNA_mapping.csv")
lnc = lnc[['lncRNA_id']].drop_duplicates()

# Key miRNAs (one column, no header)
mse = pd.read_csv("human_esm.csv", header=None)

# lncRNA-miRNA interaction table
# Column 0: miRNA_id
# Column 1: lncRNA_id
lmi = pd.read_csv("human_lmi.csv")

# =========================
# 2. Prepare reference sets
# =========================

# Set of all lncRNAs
all_lnc_set = set(lnc['lncRNA_id'])

# Set of key miRNAs
key_mir_set = set(mse.iloc[:, 0])

# =========================
# 3. Calculate lncRNA-level statistics
# =========================

# Mark whether each miRNA in the interaction table is a key miRNA
lmi['is_key_miRNA'] = lmi.iloc[:, 0].isin(key_mir_set).astype(int)

# Aggregate at lncRNA level
lnc_stat_df = (
    lmi.groupby(lmi.columns[1])
    .agg(
        total_miRNA=(lmi.columns[0], 'count'),
        key_miRNA=('is_key_miRNA', 'sum')
    )
    .reset_index()
)

# Rename the grouped lncRNA column to a standard name
lnc_stat_df = lnc_stat_df.rename(columns={lmi.columns[1]: 'lncRNA_id'})

# Define whether each lncRNA interacts with at least one known key miRNA
lnc_stat_df['has_known_key_miRNA'] = (lnc_stat_df['key_miRNA'] > 0).astype(int)

# Set of lncRNAs that have at least one interaction in lmi
network_lnc_set = set(lnc_stat_df['lncRNA_id'])

# =========================
# 4. Compare essential lncRNAs vs background lncRNAs for each tissue
# =========================

results = []

for t in ['heart', 'lung', 'stomach']:
    # Read tissue-specific essential lncRNAs
    ess_lnc = pd.read_csv(
        f'../ess_number/filtered/human/BC_top40pct_human_{t}_esslnc.csv',
        header=None
    )
    ess_set = set(ess_lnc.iloc[:, 0])

    # Background = all non-essential lncRNAs
    bg_set = all_lnc_set - ess_set

    # Only keep lncRNAs that have interaction records in lmi
    ess_valid = ess_set & network_lnc_set
    bg_valid = bg_set & network_lnc_set

    # Extract lncRNA-level records for the two groups
    ess_df = lnc_stat_df[lnc_stat_df['lncRNA_id'].isin(ess_valid)]
    bg_df = lnc_stat_df[lnc_stat_df['lncRNA_id'].isin(bg_valid)]

    # Build 2x2 contingency table
    #                has_known_key_miRNA   no_known_key_miRNA
    # essential      a                     b
    # background     c                     d
    a = (ess_df['has_known_key_miRNA'] == 1).sum()
    b = (ess_df['has_known_key_miRNA'] == 0).sum()
    c = (bg_df['has_known_key_miRNA'] == 1).sum()
    d = (bg_df['has_known_key_miRNA'] == 0).sum()

    contingency_table = [[a, b], [c, d]]

    # Chi-square test
    chi2_stat, chi2_p_value, dof, expected = chi2_contingency(contingency_table)

    # Fisher's exact test
    # greater: essential lncRNAs are more likely to have >=1 known key miRNA interaction
    fisher_or_greater, fisher_p_greater = fisher_exact(contingency_table, alternative='greater')

    # two-sided Fisher's exact test
    fisher_or_two_sided, fisher_p_two_sided = fisher_exact(contingency_table, alternative='two-sided')

    # Save summary statistics
    results.append({
        'tissue': t,

        'n_ess_total': len(ess_set),
        'n_ess_used': len(ess_valid),
        'n_bg_total': len(bg_set),
        'n_bg_used': len(bg_valid),

        'ess_with_known_key_miRNA': a,
        'ess_without_known_key_miRNA': b,
        'bg_with_known_key_miRNA': c,
        'bg_without_known_key_miRNA': d,

        'prop_ess_with_known_key_miRNA': a / len(ess_valid) if len(ess_valid) > 0 else None,
        'prop_bg_with_known_key_miRNA': c / len(bg_valid) if len(bg_valid) > 0 else None,

        'chi2_stat': chi2_stat,
        'chi2_p_value': chi2_p_value,
        'chi2_dof': dof,

        'fisher_odds_ratio': fisher_or_greater,
        'fisher_p_greater': fisher_p_greater,
        'fisher_p_two_sided': fisher_p_two_sided
    })

    # Print contingency table and expected counts for checking
    print(f"\n===== {t} =====")
    print("Contingency table:")
    print(pd.DataFrame(
        contingency_table,
        index=['essential', 'background'],
        columns=['has_known_key_miRNA', 'no_known_key_miRNA']
    ))

    print("\nExpected counts from chi-square:")
    print(pd.DataFrame(
        expected,
        index=['essential', 'background'],
        columns=['has_known_key_miRNA', 'no_known_key_miRNA']
    ))

    print(f"\nChi-square p-value: {chi2_p_value}")
    print(f"Fisher greater p-value: {fisher_p_greater}")
    print(f"Fisher two-sided p-value: {fisher_p_two_sided}")
    print(f"Fisher odds ratio: {fisher_or_greater}")

# =========================
# 5. Save results
# =========================

results_df = pd.DataFrame(results)
results_df.to_csv("human_ess_vs_background_has_known_key_miRNA_chi2_fisher.csv", index=False)

print("\nFinal results:")
print(results_df)



===== heart =====
Contingency table:
            has_known_key_miRNA  no_known_key_miRNA
essential                   637                2103
background                 1010                4335

Expected counts from chi-square:
            has_known_key_miRNA  no_known_key_miRNA
essential            558.166976         2181.833024
background          1088.833024         4256.166976

Chi-square p-value: 4.882802219546294e-06
Fisher greater p-value: 2.855477224648723e-06
Fisher two-sided p-value: 5.276290892553321e-06
Fisher odds ratio: 1.3000734452903207

===== lung =====
Contingency table:
            has_known_key_miRNA  no_known_key_miRNA
essential                   602                2044
background                 1045                4394

Expected counts from chi-square:
            has_known_key_miRNA  no_known_key_miRNA
essential            539.018182         2106.981818
background          1107.981818         4331.018182

Chi-square p-value: 0.000235966967552684
Fisher greater p

In [20]:
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

# =========================
# 1. Load input files
# =========================

# All lncRNAs
lnc = pd.read_csv("../../data/LPI/mouse/lncRNA_mapping.csv")
lnc = lnc[['lncRNA_id']].drop_duplicates()

# Key miRNAs (one column, no header)
mse = pd.read_csv("mouse_esm.csv", header=None)

# lncRNA-miRNA interaction table
# Column 0: miRNA_id
# Column 1: lncRNA_id
lmi = pd.read_csv("mouse_lmi.csv")

# =========================
# 2. Prepare reference sets
# =========================

# Set of all lncRNAs
all_lnc_set = set(lnc['lncRNA_id'])

# Set of key miRNAs
key_mir_set = set(mse.iloc[:, 0])

# =========================
# 3. Calculate lncRNA-level statistics
# =========================

# Mark whether each miRNA in the interaction table is a key miRNA
lmi['is_key_miRNA'] = lmi.iloc[:, 0].isin(key_mir_set).astype(int)

# Aggregate at lncRNA level
lnc_stat_df = (
    lmi.groupby(lmi.columns[1])
    .agg(
        total_miRNA=(lmi.columns[0], 'count'),
        key_miRNA=('is_key_miRNA', 'sum')
    )
    .reset_index()
)

# Rename the grouped lncRNA column to a standard name
lnc_stat_df = lnc_stat_df.rename(columns={lmi.columns[1]: 'lncRNA_id'})

# Define whether each lncRNA interacts with at least one known key miRNA
lnc_stat_df['has_known_key_miRNA'] = (lnc_stat_df['key_miRNA'] > 0).astype(int)

# Set of lncRNAs that have at least one interaction in lmi
network_lnc_set = set(lnc_stat_df['lncRNA_id'])

# =========================
# 4. Compare essential lncRNAs vs background lncRNAs for each tissue
# =========================

results = []

for t in ['heart', 'lung', 'brain']:
    # Read tissue-specific essential lncRNAs
    ess_lnc = pd.read_csv(
        f'../ess_number/filtered/mouse/BC_top60pct_mouse_{t}_esslnc.csv',
        header=None
    )
    ess_set = set(ess_lnc.iloc[:, 0])

    # Background = all non-essential lncRNAs
    bg_set = all_lnc_set - ess_set

    # Only keep lncRNAs that have interaction records in lmi
    ess_valid = ess_set & network_lnc_set
    bg_valid = bg_set & network_lnc_set

    # Extract lncRNA-level records for the two groups
    ess_df = lnc_stat_df[lnc_stat_df['lncRNA_id'].isin(ess_valid)]
    bg_df = lnc_stat_df[lnc_stat_df['lncRNA_id'].isin(bg_valid)]

    # Build 2x2 contingency table
    #                has_known_key_miRNA   no_known_key_miRNA
    # essential      a                     b
    # background     c                     d
    a = (ess_df['has_known_key_miRNA'] == 1).sum()
    b = (ess_df['has_known_key_miRNA'] == 0).sum()
    c = (bg_df['has_known_key_miRNA'] == 1).sum()
    d = (bg_df['has_known_key_miRNA'] == 0).sum()

    contingency_table = [[a, b], [c, d]]

    # Chi-square test
    chi2_stat, chi2_p_value, dof, expected = chi2_contingency(contingency_table)

    # Fisher's exact test
    # greater: essential lncRNAs are more likely to have >=1 known key miRNA interaction
    fisher_or_greater, fisher_p_greater = fisher_exact(contingency_table, alternative='greater')

    # two-sided Fisher's exact test
    fisher_or_two_sided, fisher_p_two_sided = fisher_exact(contingency_table, alternative='two-sided')

    # Save summary statistics
    results.append({
        'tissue': t,

        'n_ess_total': len(ess_set),
        'n_ess_used': len(ess_valid),
        'n_bg_total': len(bg_set),
        'n_bg_used': len(bg_valid),

        'ess_with_known_key_miRNA': a,
        'ess_without_known_key_miRNA': b,
        'bg_with_known_key_miRNA': c,
        'bg_without_known_key_miRNA': d,

        'prop_ess_with_known_key_miRNA': a / len(ess_valid) if len(ess_valid) > 0 else None,
        'prop_bg_with_known_key_miRNA': c / len(bg_valid) if len(bg_valid) > 0 else None,

        'chi2_stat': chi2_stat,
        'chi2_p_value': chi2_p_value,
        'chi2_dof': dof,

        'fisher_odds_ratio': fisher_or_greater,
        'fisher_p_greater': fisher_p_greater,
        'fisher_p_two_sided': fisher_p_two_sided
    })

    # Print contingency table and expected counts for checking
    print(f"\n===== {t} =====")
    print("Contingency table:")
    print(pd.DataFrame(
        contingency_table,
        index=['essential', 'background'],
        columns=['has_known_key_miRNA', 'no_known_key_miRNA']
    ))

    print("\nExpected counts from chi-square:")
    print(pd.DataFrame(
        expected,
        index=['essential', 'background'],
        columns=['has_known_key_miRNA', 'no_known_key_miRNA']
    ))

    print(f"\nChi-square p-value: {chi2_p_value}")
    print(f"Fisher greater p-value: {fisher_p_greater}")
    print(f"Fisher two-sided p-value: {fisher_p_two_sided}")
    print(f"Fisher odds ratio: {fisher_or_greater}")

# =========================
# 5. Save results
# =========================

results_df = pd.DataFrame(results)
results_df.to_csv("mouse_ess_vs_background_has_known_key_miRNA_chi2_fisher.csv", index=False)

print("\nFinal results:")
print(results_df)



===== heart =====
Contingency table:
            has_known_key_miRNA  no_known_key_miRNA
essential                   153                 101
background                  578                 321

Expected counts from chi-square:
            has_known_key_miRNA  no_known_key_miRNA
essential            161.035559           92.964441
background           569.964441          329.035559

Chi-square p-value: 0.26631094009448303
Fisher greater p-value: 0.8956534445254395
Fisher two-sided p-value: 0.23881012171541383
Fisher odds ratio: 0.8412929528246942

===== lung =====
Contingency table:
            has_known_key_miRNA  no_known_key_miRNA
essential                   265                 149
background                  466                 273

Expected counts from chi-square:
            has_known_key_miRNA  no_known_key_miRNA
essential            262.475282          151.524718
background           468.524718          270.475282

Chi-square p-value: 0.7963832839556908
Fisher greater p-value: 0

In [21]:
import pandas as pd
from scipy.stats import mannwhitneyu

# =========================
# 1. Load input files
# =========================

# All lncRNAs
lnc = pd.read_csv("../../data/LPI/human/lncRNA_mapping.csv")
lnc = lnc[['lncRNA_id']].drop_duplicates()

# Key miRNAs (one column, no header)
mse = pd.read_csv("human_esm.csv", header=None)

# lncRNA-miRNA interaction table
# Column 0: miRNA_id
# Column 1: lncRNA_id
lmi = pd.read_csv("human_lmi.csv")

# =========================
# 2. Prepare reference sets
# =========================

# Set of all lncRNAs
all_lnc_set = set(lnc['lncRNA_id'])

# Set of key miRNAs
key_mir_set = set(mse.iloc[:, 0])

# =========================
# 3. Calculate key miRNA count for each lncRNA
# =========================

# Mark whether each miRNA in the interaction table is a key miRNA
lmi['is_key_miRNA'] = lmi.iloc[:, 0].isin(key_mir_set).astype(int)

# Aggregate at lncRNA level
lnc_stat_df = (
    lmi.groupby(lmi.columns[1])
    .agg(
        total_miRNA=(lmi.columns[0], 'count'),
        key_miRNA=('is_key_miRNA', 'sum')
    )
    .reset_index()
)

# Rename the grouped lncRNA column to a standard name
lnc_stat_df = lnc_stat_df.rename(columns={lmi.columns[1]: 'lncRNA_id'})

# Set of lncRNAs that have at least one interaction in lmi
network_lnc_set = set(lnc_stat_df['lncRNA_id'])

# =========================
# 4. Compare essential lncRNAs vs background lncRNAs for each tissue
# =========================

results = []

for t in ['heart', 'lung', 'stomach']:
    # Read tissue-specific essential lncRNAs
    ess_lnc = pd.read_csv(
        f'../ess_number/filtered/human/BC_top40pct_human_{t}_esslnc.csv',
        header=None
    )
    ess_set = set(ess_lnc.iloc[:, 0])

    # Background = all non-essential lncRNAs
    bg_set = all_lnc_set - ess_set

    # Only keep lncRNAs that have interaction records in lmi
    ess_valid = ess_set & network_lnc_set
    bg_valid = bg_set & network_lnc_set

    # Extract key miRNA counts
    ess_key_counts = lnc_stat_df.loc[
        lnc_stat_df['lncRNA_id'].isin(ess_valid), 'key_miRNA'
    ]
    bg_key_counts = lnc_stat_df.loc[
        lnc_stat_df['lncRNA_id'].isin(bg_valid), 'key_miRNA'
    ]

    # Perform one-sided Mann-Whitney U test:
    # alternative='greater' means testing whether essential lncRNAs
    # interact with more key miRNAs than background lncRNAs
    u_stat, p_value = mannwhitneyu(
        ess_key_counts,
        bg_key_counts,
        alternative='greater'
    )

    # Save summary statistics
    results.append({
        'tissue': t,
        'n_ess_total': len(ess_set),
        'n_ess_used': len(ess_valid),
        'n_bg_total': len(bg_set),
        'n_bg_used': len(bg_valid),
        'mean_key_ess': ess_key_counts.mean(),
        'median_key_ess': ess_key_counts.median(),
        'mean_key_bg': bg_key_counts.mean(),
        'median_key_bg': bg_key_counts.median(),
        'u_stat': u_stat,
        'p_value': p_value
    })

# =========================
# 5. Save results
# =========================

results_df = pd.DataFrame(results)
results_df.to_csv("human_ess_vs_background_key_miRNA_mwu.csv", index=False)

print(results_df)


    tissue  n_ess_total  n_ess_used  n_bg_total  n_bg_used  mean_key_ess  \
0    heart         6727        2740       28648       5345      0.330292   
1     lung         6471        2646       28904       5439      0.321995   
2  stomach         6063        2441       29312       5644      0.310528   

   median_key_ess  mean_key_bg  median_key_bg     u_stat       p_value  
0             0.0     0.228438            0.0  7684014.0  1.044928e-07  
1             0.0     0.234234            0.0  7488335.0  1.120973e-05  
2             0.0     0.242381            0.0  7093630.5  1.189962e-03  


In [24]:
import pandas as pd
from scipy.stats import mannwhitneyu

# =========================
# 1. Load input files
# =========================

# All lncRNAs
lnc = pd.read_csv("../../data/LPI/mouse/lncRNA_mapping.csv")
lnc = lnc[['lncRNA_id']].drop_duplicates()

# Key miRNAs (one column, no header)
mse = pd.read_csv("mouse_esm.csv", header=None)

# lncRNA-miRNA interaction table
# Column 0: miRNA_id
# Column 1: lncRNA_id
lmi = pd.read_csv("mouse_lmi.csv")

# =========================
# 2. Prepare reference sets
# =========================

# Set of all lncRNAs
all_lnc_set = set(lnc['lncRNA_id'])

# Set of key miRNAs
key_mir_set = set(mse.iloc[:, 0])

# =========================
# 3. Calculate key miRNA count for each lncRNA
# =========================

# Mark whether each miRNA in the interaction table is a key miRNA
lmi['is_key_miRNA'] = lmi.iloc[:, 0].isin(key_mir_set).astype(int)

# Aggregate at lncRNA level
lnc_stat_df = (
    lmi.groupby(lmi.columns[1])
    .agg(
        total_miRNA=(lmi.columns[0], 'count'),
        key_miRNA=('is_key_miRNA', 'sum')
    )
    .reset_index()
)

# Rename the grouped lncRNA column to a standard name
lnc_stat_df = lnc_stat_df.rename(columns={lmi.columns[1]: 'lncRNA_id'})

# Set of lncRNAs that have at least one interaction in lmi
network_lnc_set = set(lnc_stat_df['lncRNA_id'])

# =========================
# 4. Compare essential lncRNAs vs background lncRNAs for each tissue
# =========================

results = []

for t in ['heart', 'lung', 'brain']:
    # Read tissue-specific essential lncRNAs
    ess_lnc = pd.read_csv(
        f'../ess_number/filtered/mouse/BC_top60pct_mouse_{t}_esslnc.csv',
        header=None
    )
    ess_set = set(ess_lnc.iloc[:, 0])

    # Background = all non-essential lncRNAs
    bg_set = all_lnc_set - ess_set

    # Only keep lncRNAs that have interaction records in lmi
    ess_valid = ess_set & network_lnc_set
    bg_valid = bg_set & network_lnc_set

    # Extract key miRNA counts
    ess_key_counts = lnc_stat_df.loc[
        lnc_stat_df['lncRNA_id'].isin(ess_valid), 'key_miRNA'
    ]
    bg_key_counts = lnc_stat_df.loc[
        lnc_stat_df['lncRNA_id'].isin(bg_valid), 'key_miRNA'
    ]

    # Perform one-sided Mann-Whitney U test:
    # alternative='greater' means testing whether essential lncRNAs
    # interact with more key miRNAs than background lncRNAs
    u_stat, p_value = mannwhitneyu(
        ess_key_counts,
        bg_key_counts,
        alternative='greater'
    )

    # Save summary statistics
    results.append({
        'tissue': t,
        'n_ess_total': len(ess_set),
        'n_ess_used': len(ess_valid),
        'n_bg_total': len(bg_set),
        'n_bg_used': len(bg_valid),
        'mean_key_ess': ess_key_counts.mean(),
        'median_key_ess': ess_key_counts.median(),
        'mean_key_bg': bg_key_counts.mean(),
        'median_key_bg': bg_key_counts.median(),
        'u_stat': u_stat,
        'p_value': p_value
    })

# =========================
# 5. Save results
# =========================

results_df = pd.DataFrame(results)
results_df.to_csv("mouse_ess_vs_background_key_miRNA_mwu.csv", index=False)

print(results_df)


  tissue  n_ess_total  n_ess_used  n_bg_total  n_bg_used  mean_key_ess  \
0  heart         4154         254       24871        899      2.814961   
1   lung         6202         414       22823        739      2.927536   
2  brain         4243         324       24782        829      3.098765   

   median_key_ess  mean_key_bg  median_key_bg    u_stat   p_value  
0             1.0      2.27030            1.0  115997.0  0.343713  
1             1.0      2.08931            1.0  162265.5  0.038259  
2             1.0      2.11339            1.0  139716.5  0.135175  
